In [ ]:
%load_ext tensorboard

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DATASET_FILE = 'jetbot_dataset_2025-12-05_13-50-12.zip'  # <-- change if needed
DATASET_DIR = 'dataset'
DATASET_ZIP = os.path.join(DATASET_DIR, DATASET_FILE)

!ls /content/drive/MyDrive/dataset/

In [ ]:
!rm -rf dataset_root
!cp '/content/drive/MyDrive/{DATASET_ZIP}' ./
!unzip -q $DATASET_FILE
!mkdir dataset_root
!mv $DATASET_DIR './dataset_root'

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets
import torchvision.transforms.v2 as transforms  # Updated to v2 to avoid deprecations
from torchvision.utils import save_image
from torch.utils.tensorboard import SummaryWriter  # Updated to native TensorBoard
from IPython.display import Image, display

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
bs = 64

dataset = datasets.ImageFolder(
    root='./dataset_root',
    transform=transforms.Compose([
        transforms.Resize((120, 160)),
        transforms.CenterCrop((80, 160)),  # Updated to CenterCrop for central region focus
        transforms.ToTensor()
    ])
)

dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=bs,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

len(dataset.imgs), len(dataloader)

In [ ]:
latent_dim = 32

class VAE(nn.Module):
    def __init__(self):
        super(VAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),  # 3x80x160 -> 32x40x80
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),  # -> 64x20x40
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),  # -> 128x10x20
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),  # -> 256x5x10
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(256 * 5 * 10, latent_dim)
        self.fc_logvar = nn.Linear(256 * 5 * 10, latent_dim)
        self.decoder_input = nn.Linear(latent_dim, 256 * 5 * 10)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),  # 256x5x10 -> 128x10x20
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),  # -> 64x20x40
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),  # -> 32x40x80
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),  # -> 3x80x160
            nn.Sigmoid(),
        )

    def encode(self, x):
        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        x = self.decoder_input(z)
        x = x.view(x.size(0), 256, 5, 10)
        x = self.decoder(x)
        return x

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

In [ ]:
def loss_function(recon_x, x, mu, logvar, beta=1.0):
    BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + beta * KLD

In [ ]:
model = VAE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
writer = SummaryWriter('runs/vae_experiment')  # For TensorBoard logging

num_epochs = 10  # Adjust as needed

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for batch_idx, (data, _) in enumerate(dataloader):
        data = data.to(device)
        optimizer.zero_grad()
        recon_batch, mu, logvar = model(data)
        loss = loss_function(recon_batch, data, mu, logvar)
        loss.backward()
        train_loss += loss.item()
        optimizer.step()
    avg_loss = train_loss / len(dataloader.dataset)
    print(f'Epoch {epoch + 1}: Average loss: {avg_loss:.4f}')
    writer.add_scalar('Loss/train', avg_loss, epoch)

In [ ]:
# Save the model 
torch.save(model.state_dict(), 'vae.torch') 
!cp vae.torch '/content/drive/My Drive/vae.torch'

In [ ]:
# Log embeddings for TensorBoard Projector 
model.eval() 
features_list = [] 
images_list = [] 
with torch.no_grad(): 
    for data, _ in dataloader: 
        data = data.to(device) 
        mu, _ = model.encode(data) 
        features_list.append(mu.cpu()) 
        images_list.append(data.cpu()) 
features = torch.cat(features_list) 
images = torch.cat(images_list)

# Subsample if too many
max_samples = 5000
if len(features) > max_samples:
    idx = torch.randperm(len(features))[:max_samples]
    features = features[idx]
    images = images[idx]

# Resize images to smaller thumbnails to avoid large sprite
thumbnail_size = (32, 64) # Smaller size, preserving aspect ratio
resize_transform = transforms.Resize(thumbnail_size)
images_resized = resize_transform(images)

writer.add_embedding(features, label_img=images_resized)

In [ ]:
%tensorboard --logdir runs

In [ ]:
def show_reconstructions(model, dataloader):
    model.eval()
    with torch.no_grad():
        data, _ = next(iter(dataloader))
        data = data.to(device)
        recon, _, _ = model(data)
        save_image(data.cpu()[:8], 'original.png')
        save_image(recon.cpu()[:8], 'recon.png')
        display(Image('original.png'))
        display(Image('recon.png'))

show_reconstructions(model, dataloader)

In [ ]:
!pip install torchmetrics[image]

import torch
import torch.nn.functional as F
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
import numpy as np
from torchmetrics.image.fid import FrechetInceptionDistance

# Assuming model, device, and dataloader are already defined from earlier cells
# If not, reload them here

model.eval()
psnr = PeakSignalNoiseRatio(data_range=1.0).to(device)
ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
fid = FrechetInceptionDistance().to(device)

mse_total = 0
psnr_total = 0
ssim_total = 0
num_batches = 0

with torch.no_grad():
    for data, _ in dataloader:
        data = data.to(device)
        recon, _, _ = model(data)
        
        # MSE (Mean Squared Error)
        mse = F.mse_loss(recon, data, reduction='mean')
        mse_total += mse.item()
        
        # PSNR
        psnr_total += psnr(recon, data).item()
        
        # SSIM
        ssim_total += ssim(recon, data).item()
        
        # FID update
        data_uint8 = (data * 255).to(torch.uint8)
        recon_uint8 = (recon * 255).to(torch.uint8)
        fid.update(data_uint8, real=True)
        fid.update(recon_uint8, real=False)
        
        num_batches += 1

avg_mse = mse_total / num_batches
avg_psnr = psnr_total / num_batches
avg_ssim = ssim_total / num_batches
fid_score = fid.compute().item()

print(f'Average MSE: {avg_mse:.4f}')
print(f'Average PSNR: {avg_psnr:.4f}')
print(f'Average SSIM: {avg_ssim:.4f}')
print(f'FID Score: {fid_score:.4f}')